# Entrenamiento del modelo BTCUSDT en Google Colab con GPU T4

Evidencia obligatoria de T4 (protocolo, sección 4): *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

Este notebook clona el repositorio, verifica la T4, entrena con **los mismos datos públicos de MEXC y la misma metodología** que `backend/training/train.py` y guarda los artefactos versionados (`model.pt`, `scaler.joblib`, `model_card.json`) en Google Drive. Si no hay T4, el script **se detiene** (`--require-t4`).

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "No hay T4: detengan y reprogramen la sesión"
x = torch.randn(1024, 1024, device="cuda"); print("Operación en", (x @ x).device, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mexc-trading-agent/models_t4'

In [ ]:
!git clone https://github.com/Julitocrack/mexc-trading-agent.git
%cd mexc-trading-agent/backend
!pip -q install fastapi httpx python-dotenv joblib

In [ ]:
!python -m training.train --require-t4 --out "$OUT" --data-dir "$OUT/data"

In [ ]:
import json
card = json.load(open(f"{OUT}/model_card.json"))
print(json.dumps({k: card[k] for k in ("version", "trained_at_utc", "signal_threshold")}, indent=2))
print(json.dumps({k: card["training"][k] for k in ("device", "gpu", "torch", "seed")}, indent=2))
print(card["training"]["nvidia_smi"][:600])

Para usar estos artefactos en el agente, copien `model.pt`, `scaler.joblib`, `model_card.json` y `curvas.csv` de `models_t4/` a `backend/models/` y reinicien el backend. El backend verifica la huella SHA-256 del modelo y el contrato de variables antes de cargarlo.